In [1]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="mesolitica/Malaysian-STT-Whisper-Stage2", 
#     repo_type="dataset",
#     allow_patterns="data/*.parquet",
#     local_dir="./Malaysian-STT-Whisper-Stage2"
# )

In [2]:
from glob import glob
import pandas as pd
import re
from collections import defaultdict
from tqdm import tqdm

In [3]:
files = glob('Malaysian-STT-Whisper-Stage2/data/*.parquet')
files = [f for f in files if 'noise' not in f and 'audioset' not in f]
len(files)

26

In [7]:
audio_filenames = set()
for f in tqdm(files):
    df = pd.read_parquet(f)
    audio_filenames.update(set(df['audio_filename'].tolist()))

100%|██████████| 26/26 [00:12<00:00,  2.12it/s]


In [8]:
len(audio_filenames)

4823180

In [12]:
import os

not_exists = [f for f in tqdm(audio_filenames) if not os.path.exists(f)]
len(not_exists)

100%|██████████| 4823180/4823180 [00:21<00:00, 223114.61it/s]


635

In [15]:
import json

with open('synthetic-files.json', 'w') as fopen:
    json.dump(list(audio_filenames), fopen)

In [ ]:
pairs = defaultdict(dict)
for f in files:
    if '_word' in f:
        split = 'word'
        splitted = f.split('_word')
    else:
        split = 'segment'
        splitted = f.split('_segment')
    pairs[splitted[0]][split] = f

In [ ]:
filenames = defaultdict(dict)

for k, v in pairs.items():
    segment = pd.read_parquet(v['segment'])
    word = pd.read_parquet(v['word'])
    
    for i in tqdm(range(len(segment))):
        filenames[segment.iloc[i]['audio_filename']]['segment'] = segment.iloc[i]['new_text']
    
    for i in tqdm(range(len(word))):
        filenames[word.iloc[i]['audio_filename']]['word'] = word.iloc[i]['new_text']

In [ ]:
file_names = list(filenames.keys())
file_names[0]

In [ ]:
import json

with open('group-filenames.json', 'w') as fopen:
    json.dump(filenames, fopen)